# Joins and Multi-Table Queries

The explainer explained why databases split data across multiple tables: normalisation prevents the update, insertion, and deletion anomalies that plague single-table designs. Each table stores one type of thing, and primary keys and foreign keys create the links between them.

The trade-off is that no single table has everything we need. The `ridership` table knows how many passengers boarded, but not the station name. The `incidents` table records delays, but not which line was affected. To answer the transport authority's real questions, we need to reconnect the tables. That is what joins do.

If you have used VLOOKUP in a spreadsheet to pull a value from one sheet into another using a shared column, you already understand the core idea. A SQL join does the same thing, but it works across entire tables at once.

By the end of this notebook we will be able to:

- Combine rows from two tables with **`INNER JOIN`**
- Include unmatched rows with **`LEFT JOIN`**
- Join three or more tables in a single query
- Use **table aliases** to keep queries readable

---

## Setup

In [ ]:
%pip install -q jupysql

In [ ]:
%load_ext sql

In [ ]:
%sql sqlite:///../data/metro_transit.sqlite

---

## 1. INNER JOIN

An `INNER JOIN` matches rows from two tables using a shared column. It returns only the rows where a match exists in both tables. If a row in one table has no corresponding row in the other, it is excluded from the result.

```sql
SELECT columns
FROM table_a
INNER JOIN table_b ON table_a.key = table_b.key;
```

The `ON` clause specifies which columns to match. These are the primary key and foreign key columns from the explainer.

### Table aliases

When joining tables, column names can be ambiguous. Both `ridership` and `stations` have a `station_id` column, so the database does not know which one we mean. **Table aliases** give each table a short name we can use as a prefix:

```sql
FROM ridership r
INNER JOIN stations s ON r.station_id = s.station_id
```

`r` is the alias for `ridership`, `s` for `stations`. We then write `r.station_id` or `s.name` to be specific about which table we are referring to.

### Ridership with station names

The `ridership` table contains `station_id` but not the station name. On its own, a list of station IDs is not useful in a report. To get the name, join to the `stations` table using the foreign key:

In [ ]:
%%sql
SELECT s.name, s.line, r.date, r.hour, r.passenger_count
FROM ridership r
INNER JOIN stations s ON r.station_id = s.station_id
ORDER BY r.passenger_count DESC
LIMIT 10;

Those are the ten highest single-hour passenger counts in the database, now shown with human-readable station names and lines instead of bare IDs.

We can combine joins with the aggregation we learned in the previous notebook. For example, total ridership per station:

In [ ]:
%%sql
SELECT s.name, s.line, SUM(r.passenger_count) AS total_passengers
FROM ridership r
INNER JOIN stations s ON r.station_id = s.station_id
GROUP BY s.station_id, s.name, s.line
ORDER BY total_passengers DESC
LIMIT 10;

### Incidents with station details

The same pattern applies to the `incidents` table. It stores `station_id` but not the station name or line. To produce a useful report of major incidents, join to `stations`:

In [ ]:
%%sql
SELECT s.name, s.line, i.date, i.incident_type, i.severity, i.delay_minutes
FROM incidents i
INNER JOIN stations s ON i.station_id = s.station_id
WHERE i.severity = 'major'
ORDER BY i.delay_minutes DESC
LIMIT 10;

---

## 2. LEFT JOIN

An `INNER JOIN` only returns rows that have a match in both tables. That is fine when we know every row will match, but sometimes the absence of a match is exactly what we need to see.

The authority wants to know the incident count for every station, including stations that have had zero incidents. An `INNER JOIN` would silently exclude those zero-incident stations. A `LEFT JOIN` keeps them.

A `LEFT JOIN` returns all rows from the left table (the one after `FROM`), even if there is no matching row in the right table. Where there is no match, the right-side columns are filled with `NULL`.

In [ ]:
%%sql
SELECT s.name, s.line, COUNT(i.incident_id) AS incident_count
FROM stations s
LEFT JOIN incidents i ON s.station_id = i.station_id
GROUP BY s.station_id, s.name, s.line
ORDER BY incident_count ASC
LIMIT 15;

Stations with `incident_count = 0` have no matching rows in the incidents table. Using `COUNT(i.incident_id)` rather than `COUNT(*)` is important here: the `NULL` values from the unmatched join are not counted, giving the correct zero.

If we used `COUNT(*)` instead, those stations would show 1, because the left join still produces one row per station regardless of whether a match exists.

### Finding rows with no match

A common and useful pattern: use `LEFT JOIN` combined with `WHERE right_table.key IS NULL` to find rows in the left table that have no corresponding row on the right.

The authority wants to know which stations have a clean safety record. Which stations have never had an incident?

In [ ]:
%%sql
SELECT s.name, s.line, s.zone
FROM stations s
LEFT JOIN incidents i ON s.station_id = i.station_id
WHERE i.incident_id IS NULL
ORDER BY s.name;

These are the stations where `i.incident_id` is `NULL` after the join, meaning no incident record matched. This pattern is something we will use regularly as analysts — finding the gaps in data is often as important as summarising the data that exists.

---

## 3. Joining three or more tables

The explainer showed how the transport database's tables connect through chains of foreign keys: tap-ins link to journeys, journeys link to routes and vehicles. Real analytical questions often require following these chains across three or more tables.

We can chain multiple joins in one query. Each `JOIN` adds another table and its own `ON` condition.

The authority wants a station overview: for each station, show the staff headcount, incident count, and average delay:

In [ ]:
%%sql
SELECT
    s.name AS station_name,
    s.line,
    COUNT(DISTINCT st.staff_id) AS staff_at_station,
    COUNT(DISTINCT i.incident_id) AS incident_count,
    ROUND(AVG(i.delay_minutes), 1) AS avg_delay
FROM stations s
LEFT JOIN incidents i ON s.station_id = i.station_id
LEFT JOIN staff st ON s.station_id = st.station_id
GROUP BY s.station_id, s.name, s.line
ORDER BY incident_count DESC
LIMIT 15;

This query pulls from three tables at once: `stations` (for names), `incidents` (for delay data), and `staff` (for headcount). Using `LEFT JOIN` for both ensures stations with zero incidents or zero assigned staff still appear in the results.

Note the use of `COUNT(DISTINCT ...)`. Without `DISTINCT`, the join would inflate counts because each incident row gets paired with each staff row for the same station. This is a common pitfall when joining multiple tables that each have a one-to-many relationship with the base table.

### Ridership and station details combined

The authority wants to compare ridership across lines and zones. That requires data from both the `ridership` table and the `stations` table:

In [ ]:
%%sql
SELECT s.line, s.zone, SUM(r.passenger_count) AS total_passengers
FROM ridership r
INNER JOIN stations s ON r.station_id = s.station_id
GROUP BY s.line, s.zone
ORDER BY s.line, s.zone;

---

## 4. INNER JOIN vs LEFT JOIN: when to use which

| Use case | Join type |
|----------|----------|
| We only want rows that exist in both tables | `INNER JOIN` |
| We want all rows from the left table, even if no match exists | `LEFT JOIN` |
| We need to find rows with no related records | `LEFT JOIN` + `WHERE ... IS NULL` |

If we are unsure which to use, `LEFT JOIN` is the safer default. It never silently drops data. We can always filter out the nulls afterwards if we do not need them.

---

## Exercises

These exercises bring together joins, aggregation, and filtering. Each one reflects a question the transport authority might need answered.

### Exercise 1: Incidents per station with station names

Count the number of incidents at each station. Return the station name, line, and incident count. Order by incident count descending. Include all stations, even those with zero incidents.

In [ ]:
%%sql
-- Your query here


### Exercise 2: Staff count per station

For each station, count the number of staff assigned to it. Return the station name, line, and staff count. Order by staff count descending. Include stations with zero staff.

In [ ]:
%%sql
-- Your query here


### Exercise 3: Stations with no incidents

Use a `LEFT JOIN` and `IS NULL` to find all stations that have never had an incident. Return the station name, line, and zone.

In [ ]:
%%sql
-- Your query here


### Exercise 4: Combined query across three tables

For each station, calculate:
- Total passenger count (from `ridership`)
- Number of incidents (from `incidents`)

Return the station name, line, total passengers, and incident count. Order by total passengers descending. Limit to the top 10.

Hint: use `LEFT JOIN` for both ridership and incidents. Use `COUNT(DISTINCT i.incident_id)` and `SUM(r.passenger_count)` carefully, or consider using subqueries.

In [ ]:
%%sql
-- Your query here


---

## Summary

We can now query across multiple tables, which is where SQL becomes genuinely powerful. In this notebook we:

- Used `INNER JOIN` to combine rows from two tables where a key matches
- Used `LEFT JOIN` to include all rows from the left table, filling in `NULL` where no match exists
- Found rows with no related records using `LEFT JOIN` + `WHERE ... IS NULL`
- Joined three tables in a single query with chained `JOIN` clauses
- Used **table aliases** to keep multi-table queries concise and readable

These three notebooks have covered the SQL fundamentals we need for data analysis: selecting, filtering, aggregating, and joining. With these tools, we can query almost any relational database we encounter. The concepts map directly to what we already know from spreadsheets — `WHERE` is a filter, `GROUP BY` is a pivot table, and `JOIN` is VLOOKUP — but SQL handles them at a scale that spreadsheets cannot.